# exp001 run002 내부 표현과 attention 분석

**CPU 런타임을 사용합니다. 모델 가중치나 GPU가 필요하지 않습니다.** 코드·metadata·데이터셋은 GitHub에서 받고, 원시 활성값 archive는 본인의 Google Drive에서 읽습니다. manifest와 작은 metadata ZIP을 먼저 로컬 cache로 복사·검증한 뒤 10문항 결과를 확인합니다. 이후 선택한 **한 문항** ZIP만 추가로 복사·검증·추출합니다. 전체 약 4GB 원시 결과를 자동 복사하지 않습니다.

이 notebook은 sampling 기본값 상속을 수정한 **수집 프로토콜 1.1 / run002**만 기본 허용합니다. 이전 run001은 greedy baseline에서 제외합니다. 아래 분석은 실제 생성 답을 causal replay한 기록이며 정답 prefix를 강제한 조건이 아닙니다.


In [ ]:
# CPU 런타임의 기본 패키지를 사용합니다. 없는 패키지만 설치합니다.
import importlib.util
import subprocess
import sys
packages = {"numpy": "numpy", "matplotlib": "matplotlib", "PIL": "pillow", "pandas": "pandas"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "torch", "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
import torch
print("Torch:", torch.__version__, "| Analysis device: CPU")


In [ ]:
from pathlib import Path
import json
import os
REPO = Path("/content/mllm-line-tracing")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/seungjoolee24/mllm-line-tracing.git", str(REPO)], check=True)
EXPERIMENT = REPO / "experiments/exp001_qwen3vl4b_trace10"
assert (EXPERIMENT / "analysis.py").exists(), "현재 checkout에 analysis.py가 없습니다. 최신 분석 notebook의 commit을 사용하세요."
DATASET = EXPERIMENT / "dataset"
sys.path.insert(0, str(EXPERIMENT))
from analysis import DriveBundle, CaseAnalysis, overview
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
print("Analysis source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())


## 1 Drive 연결과 metadata 검증

아래 Drive mount cell은 Google의 표준 인증·접근 동의 화면을 엽니다. 결과 archive가 보관된 본인 Drive를 연결하고 기본 폴더 경로를 확인하세요. manifest SHA256을 고정한 뒤 metadata ZIP의 SHA256·파일 크기를 검사하고 로컬 cache에 풉니다. Tensor는 읽을 때 다시 hash·shape·dtype을 확인합니다.

Drive 폴더에는 `artifact_manifest.json`, `run002_metadata.zip`, 문항별 ZIP 10개가 있어야 합니다. 빠졌거나 hash가 다르면 명확하게 실패하며 임의의 다른 run으로 대체하지 않습니다.


In [ ]:
# 실행하면 Google Drive의 표준 연결·접근 동의 화면이 표시됩니다.
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ARTIFACT_DIR = "/content/drive/MyDrive/mllm-line-tracing/experiments/exp001_qwen3vl4b_trace10/run002_artifacts" #@param {type:"string"}
EXPECTED_MANIFEST_SHA256 = "7d86f02d1d52ab50a40a1abe0028334e1773f5a4b854bb645efe7f7ddca35e5e" #@param {type:"string"}
CACHE = Path("/content/exp001_analysis_cache")
bundle = DriveBundle(DRIVE_ARTIFACT_DIR, CACHE, expected_manifest_sha256=EXPECTED_MANIFEST_SHA256)
RUN = bundle.metadata()
print("Verified manifest SHA256:", bundle.manifest_sha256)
print("Restored local metadata:", RUN)
display(pd.DataFrame(bundle.inventory()))


In [ ]:
table = pd.DataFrame(overview(RUN, DATASET))
display(table)
print(f"Exact match: {int(table.exact_match.sum())}/{len(table)}")
print("All raw argmax aligned:", bool(table.raw_argmax_alignment.all()))
print("All replay argmax aligned:", bool(table.replay_argmax_alignment.all()))
assert len(table) == bundle.manifest["case_count"] == 10
assert table.raw_argmax_alignment.all() and table.replay_argmax_alignment.all()


## 2 한 문항만 선택해 tensor 복원

기본값은 `case_03_swirl`입니다. 아래 cell은 선택한 한 문항의 ZIP만 Drive에서 로컬 cache로 복사합니다. 다른 문항을 보려면 CASE_ID를 바꿔 이 단계부터 실행하면 됩니다. 여러 문항을 모두 자동 순회하지 않습니다.


In [ ]:
CASE_ID = "case_03_swirl" #@param ["case_00_curve", "case_01_straight", "case_02_elbow", "case_03_swirl", "case_04_curve", "case_05_straight", "case_06_elbow", "case_07_swirl", "case_08_curve", "case_09_elbow"]
chosen_asset = next(row for row in bundle.inventory() if row["kind"] == "case" and row["sample_id"] == CASE_ID)
print(f"Copying selected Drive archive only: {chosen_asset['file']} ({chosen_asset['MiB']:.1f} MiB)")
bundle.case(CASE_ID)
case = CaseAnalysis(RUN, CASE_ID, DATASET)
print(json.dumps(case.mapping_summary(), indent=2))
print("Expected:", case.sample["answer"])
print("Generated:", case.meta["generation"]["raw_answer"])
display(Image.open(case.image_path))


## 3 좌표와 token 순서 확인

640×640 입력에서는 pre-merger ViT patch 1,600개, merged visual token과 decoder image 위치 400개가 기록돼야 합니다. ViT patch는 단순 전체 이미지 row-major가 아니라 **2×2 merge block-major** 순서입니다. 아래 표의 실제 index·bbox로 공간에 다시 배치합니다. 모델 전처리에서 resize가 없었던 수집 조건을 사용합니다.


In [ ]:
print("ViT tokens:", len(case.geometry["vision_tokens"]))
print("Merged tokens:", len(case.geometry["merged_tokens"]))
assert len(case.geometry["vision_tokens"]) == 1600
assert len(case.geometry["merged_tokens"]) == len(case.image_positions) == 400
display(pd.DataFrame(case.geometry["vision_tokens"]).head(12))
display(pd.DataFrame(case.geometry["merged_tokens"]).head(8))
display(pd.DataFrame(case.meta["generation"]["token_details"]["tokens"]))
display(pd.DataFrame(case.meta["generation"]["token_details"]["color_spans"]))

print("Input question token positions:")
display(pd.DataFrame([row for row in case.meta["input"]["token_table"] if row["kind"] == "question"]))


## 4 층별 token 활성값

각 tensor를 하나씩 CPU에 읽고 norm과 이전 층 대비 cosine만 작은 NumPy 배열로 남깁니다. 모든 사례의 원시 tensor를 메모리에 함께 쌓지 않습니다. norm은 원본 hidden vector의 크기이며 의미적 중요도나 인과적 기여를 뜻하지 않습니다. 마지막 vision 층의 큰 norm이 다른 층을 가리지 않도록 norm 색상은 기본 log scale이고 실제 저장 통계는 원래 값입니다. Cosine 색상은 −1에서 1로 고정합니다.

Decoder output은 **동일 인덱스의 DeepStack 주입 전**입니다. 다음 층 input에는 이전 DeepStack 주입이 포함됩니다. 따라서 초기 층의 output-to-output 차이에는 이 주입 효과도 들어갑니다. 원본 `capture_semantics`를 앞 단계에서 그대로 출력했습니다.


In [ ]:
vision_stats = case.activation_statistics("vision", "output")
case.plot_layer_statistics(vision_stats)
plt.show()
decoder_stats = case.activation_statistics("decoder", "output")
case.plot_layer_statistics(decoder_stats)
plt.show()
case.plot_group_curves(decoder_stats)
plt.show()


In [ ]:
VISION_LAYER = 12 #@param {type:"integer"}
DECODER_LAYER = 18 #@param {type:"integer"}
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
case.plot_overlay(vision_stats["norm"][VISION_LAYER], f"ViT L{VISION_LAYER} token norm", stage="vision", ax=axes[0])
case.plot_overlay(decoder_stats["norm"][DECODER_LAYER, case.image_positions], f"Decoder L{DECODER_LAYER} image-position norm", ax=axes[1])
plt.show()


## 5 생성 시점별 image attention

`predictor`는 해당 답 token을 **예측한 위치**입니다. 첫 답은 마지막 prompt 위치에서 예측됩니다. `generated`는 이미 생성된 token 자체가 입력으로 처리된 위치이므로 다음 token 예측과 연결됩니다. 두 행을 같은 시점으로 해석하지 않습니다. `question`은 입력 질문 token 위치입니다.

HEAD=-1이면 모든 head 평균, 0–31이면 개별 head입니다. 저장된 값은 전체 시퀀스에 대한 원래 softmax 확률이며 이미지 열만 다시 합이 1이 되도록 정규화하지 않습니다. 각 공간 cell의 색상은 해당 merged token의 값입니다.


In [ ]:
LAYER = 18 #@param {type:"integer"}
HEAD = -1 #@param {type:"integer"}
QUERY_KIND = "predictor" #@param ["predictor", "generated", "question"]
QUERY_INDEX = 0 #@param {type:"integer"}
head = None if HEAD == -1 else HEAD
rows = case.attention_rows(LAYER, QUERY_KIND, head)
assert 0 <= QUERY_INDEX < len(rows), f"QUERY_INDEX range: 0..{len(rows)-1}"
print("Image attention mass:", float(rows[QUERY_INDEX].sum()))
case.plot_attention(LAYER, QUERY_KIND, QUERY_INDEX, head)
plt.show()


In [ ]:
# 같은 query를 세 층에서 비교합니다. 모든 panel에 같은 color scale을 적용합니다.
LAYERS = [0, 17, 35]
fields = [case.attention_rows(layer, QUERY_KIND, head)[QUERY_INDEX] for layer in LAYERS]
maximum = max(float(values.max()) for values in fields)
fig, axes = plt.subplots(1, len(LAYERS), figsize=(17, 5))
for ax, layer, values in zip(axes, LAYERS, fields):
    case.plot_overlay(values, f"Decoder L{layer} | {QUERY_KIND}[{QUERY_INDEX}]", ax=ax, vmin=0, vmax=maximum)
plt.show()


## 6 색 출력 시점과 정답 오브젝트 순서

색 이름이 여러 BPE token으로 구성되면 해당 token의 predictor 행들을 평균합니다. 세로축은 출력에서 발견한 색 이름의 순서, 가로축은 생성기의 정답 오브젝트 순서입니다. 청록색 대각선은 출력 slot과 정답 순서가 같은 위치를 표시합니다. 출력 형식이 어긋난 경우 세로축을 실제 경로 방문 순서로 단정하지 않습니다.

오브젝트 mass는 `token attention × 해당 token cell에서 object mask가 차지하는 면적 비율`의 합입니다. **32×32 cell 안의 attention이 균일하다고 가정한 투영**이며 pixel 단위로 관측한 attention이 아닙니다. 오브젝트 영역끼리 다시 정규화하지 않습니다.


In [ ]:
case.plot_object_attention(LAYER, kind="predictor", head=head)
plt.show()
case.plot_object_attention(LAYER, kind="generated", head=head)
plt.show()
case.plot_attention_layer_profile("predictor", head)
plt.show()
case.plot_attention_layer_profile("question", head)
plt.show()


## 7 분석 요약 저장

작은 파생 통계와 표만 저장합니다. 원시 tensor를 다시 복제하지 않습니다. 복사·추출한 case ZIP과 tensor는 로컬 cache에 남으므로 추가 사례를 선택할 때 디스크 사용량을 확인하세요.


In [ ]:
EXPORT = Path("/content/exp001_analysis_exports") / CASE_ID
EXPORT.mkdir(parents=True, exist_ok=True)
table.to_csv(EXPORT / "ten_case_overview.csv", index=False)
np.savez_compressed(EXPORT / "activation_statistics.npz",
                    vision_norm=vision_stats["norm"], vision_cosine=vision_stats["cosine_previous_layer"],
                    decoder_norm=decoder_stats["norm"], decoder_cosine=decoder_stats["cosine_previous_layer"])
object_mass, generated_colors = case.object_attention(LAYER, "predictor", head)
pd.DataFrame(object_mass, index=[f"{i}:{color}" for i, color in enumerate(generated_colors)],
             columns=[f"{i}:{obj['color']}" for i, obj in enumerate(case.objects)]).to_csv(EXPORT / "object_attention_mass.csv")
provenance = {"artifact_source": "mounted Google Drive", "drive_artifact_dir": DRIVE_ARTIFACT_DIR,
              "manifest_sha256": bundle.manifest_sha256,
              "sample_id": CASE_ID, "layer": LAYER, "head": HEAD,
              "capture_semantics": case.meta["capture_semantics"],
              "collection_mode": case.meta["alignment"]["collection_mode"]}
(EXPORT / "analysis_provenance.json").write_text(json.dumps(provenance, indent=2)+"\n")
print("Saved:", EXPORT)


## 해석 범위

실제 생성 답을 causal replay한 값이며 정답 prefix를 teacher forcing한 실험이 아닙니다. Attention heatmap만으로 tracing의 인과적 메커니즘을 증명할 수 없습니다. 활성값 norm과 층간 cosine도 기술 통계입니다. 후속 masking이나 activation intervention으로 실제 기여를 확인할 수 있습니다. 10문항 결과는 이 고정된 작은 문제 집합에 대한 관찰이며 일반화 성능 추정이 아닙니다.
